# 🎓 Homework 1: Data Preprocessing & Classification Models

**Sinh viên:** Ngô Lâm Tiên  
**MSSV:** 524H0195  

---

## 📦 Dataset: Telco Customer Churn

Dataset dự đoán khách hàng rời bỏ dịch vụ viễn thông.  
Chứa đầy đủ: **numerical, categorical, ordinal, datetime (tenure), text (customerID)** features.

**Mục tiêu (Target):** `Churn` — phân loại nhị phân: `Yes` / `No`

**Nguồn:** IBM Sample Dataset / Kaggle Telco Customer Churn

---

### 📋 Mục lục
1. Import thư viện
2. Load & tạo dataset
3. Exploratory Data Analysis (EDA)
4. **Data Cleaning** (missing, duplicated, inconsistent, invalid)
5. **Feature Engineering & Conversion**
6. **Preprocessing Pipeline**
7. Chia tập Train/Test
8. **Huấn luyện & so sánh 3+ mô hình phân loại**
9. **Đánh giá mô hình** (Accuracy, Precision, Recall, F1, Confusion Matrix)
10. Kết luận


---
## Bước 1: Import thư viện


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn - Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import (StandardScaler, MinMaxScaler, LabelEncoder,
                                   OneHotEncoder, OrdinalEncoder)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Sklearn - Models
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Sklearn - Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Plot style
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
COLORS = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

print('✅ Tất cả thư viện đã được import thành công!')
import sklearn; print(f'   scikit-learn version: {sklearn.__version__}')
print(f'   pandas version     : {pd.__version__}')
print(f'   numpy version      : {np.__version__}')


---
## Bước 2: Load & tạo Dataset

Dataset **Telco Customer Churn** chứa nhiều loại feature:
- **Numerical**: `tenure`, `MonthlyCharges`, `TotalCharges`
- **Categorical (nominal)**: `gender`, `InternetService`, `Contract`, `PaymentMethod`
- **Categorical (binary)**: `PhoneService`, `OnlineSecurity`, `TechSupport`, `StreamingTV`, `StreamingMovies`, `PaperlessBilling`, `Churn`
- **Ordinal**: `ContractRank` (thêm trong feature engineering)
- **Text**: `customerID`

Ta sẽ load từ URL hoặc tạo dữ liệu mẫu với các lỗi nhân tạo để demo data cleaning.


In [ ]:
# ==============================================================
# TẠO DATASET TELCO CUSTOMER CHURN MÔ PHỎNG
# (chứa đủ: numerical, categorical, ordinal, text features
#  + các lỗi nhân tạo để demo data cleaning)
# ==============================================================

np.random.seed(42)
N = 1500

# --- Các giá trị hợp lệ ---
genders        = np.random.choice(['Male', 'Female'], N)
senior         = np.random.choice([0, 1], N, p=[0.84, 0.16])
partner        = np.random.choice(['Yes', 'No'], N)
dependents     = np.random.choice(['Yes', 'No'], N, p=[0.3, 0.7])
tenure_raw     = np.random.randint(1, 72, N).astype(float)
phone_service  = np.random.choice(['Yes', 'No'], N, p=[0.9, 0.1])
internet_svc   = np.random.choice(['DSL', 'Fiber optic', 'No'], N, p=[0.34, 0.44, 0.22])
online_sec     = np.random.choice(['Yes', 'No', 'No internet service'], N)
tech_support   = np.random.choice(['Yes', 'No', 'No internet service'], N)
streaming_tv   = np.random.choice(['Yes', 'No', 'No internet service'], N)
streaming_mov  = np.random.choice(['Yes', 'No', 'No internet service'], N)
contracts      = np.random.choice(['Month-to-month', 'One year', 'Two year'],
                                   N, p=[0.55, 0.21, 0.24])
paperless      = np.random.choice(['Yes', 'No'], N)
payment        = np.random.choice(
    ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'],
    N, p=[0.34, 0.23, 0.22, 0.21]
)
monthly        = np.random.uniform(18, 120, N).round(2)
total_charges  = (tenure_raw * monthly + np.random.normal(0, 50, N)).round(2)
total_charges  = np.clip(total_charges, 0, None)

# Nhãn phụ thuộc vào contract + charges (tạo tương quan)
churn_prob = (
    0.4 * (contracts == 'Month-to-month').astype(float)
  + 0.1 * (monthly > 80).astype(float)
  - 0.1 * (tenure_raw > 36).astype(float)
  + np.random.uniform(0, 0.3, N)
)
churn = np.where(churn_prob > 0.45, 'Yes', 'No')

# --- Customer ID (text feature) ---
cust_id = [f'{np.random.randint(1000,9999)}-{"ABCDE"[np.random.randint(5)]:s}'
            + f'{np.random.randint(10000, 99999)}' for _ in range(N)]

df = pd.DataFrame({
    'customerID'      : cust_id,
    'gender'          : genders,
    'SeniorCitizen'   : senior,
    'Partner'         : partner,
    'Dependents'      : dependents,
    'tenure'          : tenure_raw,
    'PhoneService'    : phone_service,
    'InternetService' : internet_svc,
    'OnlineSecurity'  : online_sec,
    'TechSupport'     : tech_support,
    'StreamingTV'     : streaming_tv,
    'StreamingMovies' : streaming_mov,
    'Contract'        : contracts,
    'PaperlessBilling': paperless,
    'PaymentMethod'   : payment,
    'MonthlyCharges'  : monthly,
    'TotalCharges'    : total_charges,
    'Churn'           : churn,
})

# =============================================
# INJECT LỖI NHÂN TẠO để demo data cleaning
# =============================================
rng = np.random.default_rng(0)

# 1) Missing values (~5%)
for col in ['tenure', 'MonthlyCharges', 'TotalCharges', 'gender', 'Contract']:
    idx = rng.choice(N, int(N * 0.04), replace=False)
    df.loc[idx, col] = np.nan

# 2) Duplicate rows (30 bản sao)
dup_idx = rng.choice(N, 30, replace=False)
df = pd.concat([df, df.iloc[dup_idx]], ignore_index=True)

# 3) Inconsistent casing
bad_idx = rng.choice(N, 40, replace=False)
df.loc[bad_idx, 'gender'] = df.loc[bad_idx, 'gender'].str.lower()

# 4) Invalid values
df.loc[rng.choice(N, 10, replace=False), 'tenure'] = -99   # giá trị âm
df.loc[rng.choice(N, 10, replace=False), 'MonthlyCharges'] = 9999  # outlier bất thường

print('=== DATASET GỐC (SAU KHI INJECT LỖI) ===')
print(f'Số mẫu   : {len(df):,}')
print(f'Số cột   : {df.shape[1]}')
print(f'\nPhân bố nhãn Churn:')
print(df['Churn'].value_counts())
print(f'\n5 dòng đầu:')
df.head()


---
## Bước 3: Exploratory Data Analysis (EDA)


In [ ]:
print('=== THÔNG TIN DATASET ===')
df.info()


In [ ]:
print('=== THỐNG KÊ MÔ TẢ ===')
df.describe(include='all').T


In [ ]:
# ---- Phân bố nhãn & các feature quan trọng ----
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Phân bố nhãn
cnt = df['Churn'].value_counts()
axes[0, 0].bar(cnt.index, cnt.values, color=['#2ecc71', '#e74c3c'], edgecolor='white', width=0.5)
axes[0, 0].set_title('Phân bố nhãn Churn', fontweight='bold')
axes[0, 0].set_ylabel('Số lượng')
for i, v in enumerate(cnt.values):
    axes[0, 0].text(i, v + 5, f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=10)

# 2. Tenure histogram
axes[0, 1].hist(df['tenure'].dropna(), bins=30, color='#3498db', edgecolor='white', alpha=0.85)
axes[0, 1].set_title('Phân bố Tenure (tháng)', fontweight='bold')
axes[0, 1].set_xlabel('Tháng')

# 3. MonthlyCharges
axes[0, 2].hist(df['MonthlyCharges'].dropna(), bins=30, color='#9b59b6', edgecolor='white', alpha=0.85)
axes[0, 2].set_title('Phân bố MonthlyCharges ($)', fontweight='bold')
axes[0, 2].set_xlabel('USD')

# 4. Contract
c_cnt = df['Contract'].value_counts()
axes[1, 0].barh(c_cnt.index, c_cnt.values, color=COLORS[:3], edgecolor='white')
axes[1, 0].set_title('Phân bố Contract', fontweight='bold')

# 5. Churn by Contract
ct = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
ct.plot(kind='bar', ax=axes[1, 1], color=['#2ecc71', '#e74c3c'], edgecolor='white', rot=20)
axes[1, 1].set_title('Churn Rate theo Contract (%)', fontweight='bold')
axes[1, 1].set_ylabel('%')
axes[1, 1].legend(title='Churn')

# 6. Churn by InternetService
ct2 = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100
ct2.plot(kind='bar', ax=axes[1, 2], color=['#2ecc71', '#e74c3c'], edgecolor='white', rot=20)
axes[1, 2].set_title('Churn Rate theo InternetService (%)', fontweight='bold')
axes[1, 2].set_ylabel('%')
axes[1, 2].legend(title='Churn')

plt.suptitle('📊 Exploratory Data Analysis — Telco Churn', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## Bước 4: Data Cleaning

### 4.1 Kiểm tra Missing Values


In [ ]:
print('=== 4.1 KIỂM TRA MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_df = pd.DataFrame({'Số lượng thiếu': missing, 'Tỉ lệ (%)': missing_pct})
miss_df = miss_df[miss_df['Số lượng thiếu'] > 0].sort_values('Tỉ lệ (%)', ascending=False)
print(miss_df)

# Visualize missing
fig, ax = plt.subplots(figsize=(8, 3))
miss_df['Tỉ lệ (%)'].plot(kind='bar', ax=ax, color='#e74c3c', edgecolor='white')
ax.set_title('Tỉ lệ Missing Values theo cột (%)', fontweight='bold')
ax.set_ylabel('%')
ax.axhline(y=20, color='gray', linestyle='--', label='Ngưỡng xóa cột (20%)')
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


### 4.2 Kiểm tra Duplicate Rows


In [ ]:
print('=== 4.2 KIỂM TRA DUPLICATE ROWS ===')
n_dup = df.duplicated().sum()
print(f'Số dòng trùng lặp: {n_dup}')

# Xóa duplicate
df = df.drop_duplicates().reset_index(drop=True)
print(f'Số mẫu sau khi xóa duplicate: {len(df):,}')


### 4.3 Xử lý Inconsistent Data (chuẩn hoá casing)


In [ ]:
print('=== 4.3 INCONSISTENT DATA ===')

# Kiểm tra giá trị duy nhất trong các cột categorical
cat_cols_check = ['gender', 'Contract', 'InternetService', 'PaymentMethod']
for col in cat_cols_check:
    print(f'{col}: {sorted(df[col].dropna().unique())}')

# Chuẩn hoá: title-case tất cả cột text
for col in ['gender', 'Partner', 'Dependents', 'PhoneService',
            'InternetService', 'OnlineSecurity', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod', 'Churn']:
    df[col] = df[col].str.strip().str.title() if df[col].dtype == object else df[col]

print('\nSau khi chuẩn hoá:')
for col in cat_cols_check:
    print(f'{col}: {sorted(df[col].dropna().unique())}')


### 4.4 Xử lý Invalid Values


In [ ]:
print('=== 4.4 INVALID VALUES ===')

# 1. tenure âm → set NaN
neg_tenure = (df['tenure'] < 0).sum()
print(f'Tenure âm: {neg_tenure} mẫu → thay bằng NaN')
df.loc[df['tenure'] < 0, 'tenure'] = np.nan

# 2. MonthlyCharges > 500 → bất thường → thay NaN
outlier_monthly = (df['MonthlyCharges'] > 500).sum()
print(f'MonthlyCharges > 500: {outlier_monthly} mẫu → thay bằng NaN')
df.loc[df['MonthlyCharges'] > 500, 'MonthlyCharges'] = np.nan

# 3. TotalCharges < 0 → không hợp lệ
neg_total = (df['TotalCharges'] < 0).sum()
print(f'TotalCharges < 0: {neg_total} mẫu → thay bằng NaN')
df.loc[df['TotalCharges'] < 0, 'TotalCharges'] = np.nan

print('\nMissing values sau xử lý invalid:')
print(df.isnull().sum()[df.isnull().sum() > 0])


---
## Bước 5: Feature Engineering & Conversion

Chuyển đổi từng loại feature sang format phù hợp.


In [ ]:
print('=== 5.1 XỬ LÝ TEXT FEATURE ===')
# customerID là ID thuần → không có giá trị dự đoán → drop
print(f'Ví dụ customerID: {df["customerID"].head(3).tolist()}')
print('→ Drop cột customerID (text thuần, không có giá trị dự đoán)')
df = df.drop(columns=['customerID'])

print('=== 5.2 CHUYỂN ĐỔI KIỂU DỮ LIỆU ===')
# Đảm bảo numerical đúng kiểu
df['tenure']        = pd.to_numeric(df['tenure'],        errors='coerce')
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
df['TotalCharges']  = pd.to_numeric(df['TotalCharges'],  errors='coerce')
df['SeniorCitizen'] = df['SeniorCitizen'].astype(float)

print(df.dtypes)


In [ ]:
print('=== 5.3 FEATURE ENGINEERING ===')

# --- A. ContractRank (Ordinal feature) ---
# Month-to-month < One year < Two year → ánh xạ sang 0, 1, 2
contract_order = {'Month-To-Month': 0, 'One Year': 1, 'Two Year': 2}
df['ContractRank'] = df['Contract'].map(contract_order)
print('ContractRank (ordinal):', df['ContractRank'].value_counts().to_dict())

# --- B. ChargesPerMonth (numerical feature mới) ---
df['ChargesPerMonth'] = (df['TotalCharges'] / df['tenure'].replace(0, 1)).round(2)
print('ChargesPerMonth - sample:', df['ChargesPerMonth'].head(3).tolist())

# --- C. HasServices (count dịch vụ thêm) ---
service_cols = ['OnlineSecurity', 'TechSupport', 'StreamingTV', 'StreamingMovies']
df['NumServices'] = sum(
    (df[c] == 'Yes').astype(int) for c in service_cols
)
print('NumServices dist:', df['NumServices'].value_counts().sort_index().to_dict())

# --- D. TenureGroup (bins) ---
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-1yr', '1-2yr', '2-4yr', '4+yr']
)
print('TenureGroup dist:', df['TenureGroup'].value_counts().sort_index().to_dict())

print('\nDataFrame sau feature engineering:')
print(df.shape)
df.head(3)


---
## Bước 6: Preprocessing Pipeline

Xây dựng pipeline để:
1. **Impute** missing values
2. **Encode** categorical features
3. **Scale** numerical features

**Quan trọng:** fit ONLY trên tập train, transform cả train và test.


In [ ]:
# ---- Mã hoá nhãn ----
le = LabelEncoder()
y = le.fit_transform(df['Churn'])  # No=0, Yes=1
print(f'Nhãn: {le.classes_} → {le.transform(le.classes_)}')
print(f'Phân bố: No={np.sum(y==0)}, Yes={np.sum(y==1)}')

# ---- Tách features ----
X = df.drop(columns=['Churn'])

# ---- Xác định loại cột ----
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
                  'ChargesPerMonth', 'NumServices', 'ContractRank']

binary_cols = ['SeniorCitizen']  # đã là 0/1

categorical_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
                    'InternetService', 'OnlineSecurity', 'TechSupport',
                    'StreamingTV', 'StreamingMovies', 'Contract',
                    'PaperlessBilling', 'PaymentMethod']

ordinal_cols_drop = ['TenureGroup']  # đã encode thành ContractRank + NumServices

# Chỉ giữ các cột đã định nghĩa
X_raw = X[numerical_cols + binary_cols + categorical_cols].copy()
print(f'\nFeatures sử dụng: {X_raw.shape[1]} cột')
print(X_raw.dtypes)


In [ ]:
# ========================================================
# XÂY DỰNG PREPROCESSING PIPELINE (sklearn Pipeline)
# ========================================================

# --- Pipeline cho numerical ---
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # điền median
    ('scaler',  StandardScaler())                    # chuẩn hoá
])

# --- Pipeline cho binary ---
bin_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# --- Pipeline cho categorical ---
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),   # điền mode
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# --- ColumnTransformer gộp tất cả ---
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, numerical_cols),
    ('bin', bin_pipeline, binary_cols),
    ('cat', cat_pipeline, categorical_cols),
], remainder='drop')

print('=== CẤU TRÚC PREPROCESSING PIPELINE ===')
print(preprocessor)
print('\nGiải thích:')
print('  num  → Median Impute → StandardScaler     (6 cột)')
print('  bin  → Mode Impute                         (1 cột)')
print('  cat  → Mode Impute  → One-Hot Encoding    (12 cột)')


---
## Bước 7: Chia tập Train / Test


In [ ]:
# ---- Chia 75% train, 25% test ----
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print('=== CHIA TẬP DỮ LIỆU ===')
print(f'Tổng số mẫu : {len(X_raw):,}')
print(f'Tập Train   : {len(X_train):,} ({len(X_train)/len(X_raw)*100:.0f}%)')
print(f'Tập Test    : {len(X_test):,}  ({len(X_test)/len(X_raw)*100:.0f}%)')
print(f'\nPhân bố nhãn train: No={np.sum(y_train==0)}, Yes={np.sum(y_train==1)}')
print(f'Phân bố nhãn test : No={np.sum(y_test==0)}, Yes={np.sum(y_test==1)}')

# ---- Áp dụng preprocessing (fit chỉ trên train!) ----
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

print(f'\nSau preprocessing:')
print(f'  X_train shape: {X_train_proc.shape}')
print(f'  X_test  shape: {X_test_proc.shape}')


---
## Bước 8: Huấn luyện & So sánh các mô hình phân loại

Ta sẽ huấn luyện **5 mô hình** và so sánh:

| # | Mô hình | Ghi chú |
|---|---------|----------|
| 1 | **kNN** | K-Nearest Neighbors — mô hình chính của bài |
| 2 | **Decision Tree** | Cây quyết định |
| 3 | **Naive Bayes** | Xác suất Bayes đơn giản |
| 4 | **Logistic Regression** | Hồi quy logistic |
| 5 | **Random Forest** | Ensemble — tự học thêm |


In [ ]:
# ============================================================
# HUẤN LUYỆN TẤT CẢ MÔ HÌNH
# ============================================================

models = {
    'kNN (k=7)'          : KNeighborsClassifier(n_neighbors=7, metric='euclidean'),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=8, random_state=42),
    'Naive Bayes'        : GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}

results = {}
print(f'{"Mô hình":<22} {"Accuracy":>9} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('-' * 60)

for name, model in models.items():
    model.fit(X_train_proc, y_train)
    y_pred = model.predict(X_test_proc)
    
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    cm   = confusion_matrix(y_test, y_pred)
    
    results[name] = {
        'model'    : model,
        'y_pred'   : y_pred,
        'Accuracy' : acc,
        'Precision': prec,
        'Recall'   : rec,
        'F1'       : f1,
        'CM'       : cm,
    }
    print(f'{name:<22} {acc:>9.4f} {prec:>10.4f} {rec:>8.4f} {f1:>8.4f}')

print('-' * 60)


---
## Bước 9: Đánh giá mô hình chi tiết

### 9.1 Bảng so sánh tổng hợp


In [ ]:
# ---- Bảng tổng hợp ----
metrics_df = pd.DataFrame([
    {'Model': name, 'Accuracy': v['Accuracy'], 'Precision': v['Precision'],
     'Recall': v['Recall'], 'F1-Score': v['F1']}
    for name, v in results.items()
]).set_index('Model').round(4)

print('=== BẢNG SO SÁNH CHỈ SỐ ĐÁNH GIÁ ===')
print(metrics_df.to_string())

# Highlight best
best_acc = metrics_df['Accuracy'].idxmax()
best_f1  = metrics_df['F1-Score'].idxmax()
print(f'\n🏆 Mô hình Accuracy cao nhất: {best_acc} ({metrics_df.loc[best_acc,"Accuracy"]:.4f})')
print(f'🏆 Mô hình F1-Score cao nhất : {best_f1}  ({metrics_df.loc[best_f1,"F1-Score"]:.4f})')


In [ ]:
# ---- Visualize: Bar chart so sánh các chỉ số ----
fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(len(metrics_df))
w = 0.18
metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metric_colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

for i, (metric, color) in enumerate(zip(metrics_list, metric_colors)):
    bars = ax.bar(x + i*w, metrics_df[metric], width=w, label=metric,
                  color=color, edgecolor='white', alpha=0.88)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.005,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(metrics_df.index, rotation=15, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('📊 So sánh các mô hình phân loại — Telco Churn', fontweight='bold', fontsize=13)
ax.legend(loc='upper right', framealpha=0.9)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, linewidth=1)
plt.tight_layout()
plt.show()


### 9.2 Confusion Matrix cho từng mô hình


In [ ]:
# ---- Confusion Matrix 5 mô hình ----
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

class_names = le.classes_
cmaps = ['Blues', 'Greens', 'Oranges', 'Purples', 'Reds']

for i, (name, v) in enumerate(results.items()):
    disp = ConfusionMatrixDisplay(confusion_matrix=v['CM'], display_labels=class_names)
    disp.plot(ax=axes[i], cmap=cmaps[i % len(cmaps)], colorbar=False)
    acc = v['Accuracy']
    f1  = v['F1']
    axes[i].set_title(f'{name}\nAcc={acc:.3f}  F1={f1:.3f}', fontweight='bold')

# Ẩn ô trống
axes[-1].set_visible(False)

plt.suptitle('📊 Confusion Matrix — Tất cả mô hình', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### 9.3 Classification Report chi tiết


In [ ]:
# ---- Classification Report từng mô hình ----
for name, v in results.items():
    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(classification_report(y_test, v['y_pred'],
                                 target_names=class_names, digits=4))


### 9.4 Cross-Validation Score


In [ ]:
# ---- 5-Fold Cross Validation ----
print('=== 5-FOLD CROSS VALIDATION (trên tập train, dùng X_train_proc) ===')
print(f'{"Mô hình":<22} {"CV Mean":>9} {"CV Std":>8}')
print('-' * 42)

cv_results = {}
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, v in results.items():
    cv = cross_val_score(v['model'], X_train_proc, y_train,
                         cv=kfold, scoring='f1', n_jobs=-1)
    cv_results[name] = cv
    print(f'{name:<22} {cv.mean():>9.4f} ± {cv.std():>6.4f}')

# Visualize CV
fig, ax = plt.subplots(figsize=(10, 5))
names  = list(cv_results.keys())
means  = [v.mean() for v in cv_results.values()]
stds   = [v.std()  for v in cv_results.values()]

bars = ax.barh(names, means, xerr=stds, color=COLORS, edgecolor='white',
                alpha=0.85, capsize=5)
ax.set_xlabel('F1-Score (5-Fold CV)')
ax.set_title('📊 Cross-Validation F1-Score (5-Fold)', fontweight='bold')
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.6)
for bar, mean, std in zip(bars, means, stds):
    ax.text(mean + std + 0.005, bar.get_y() + bar.get_height()/2,
            f'{mean:.3f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()


---
## Bước 10: kNN — Tìm k tối ưu


In [ ]:
# ---- Tìm k tối ưu cho kNN ----
k_values  = list(range(1, 31, 2))
train_acc = []
test_acc  = []
test_f1   = []

for k in k_values:
    knn_k = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
    knn_k.fit(X_train_proc, y_train)
    train_acc.append(accuracy_score(y_train, knn_k.predict(X_train_proc)))
    preds = knn_k.predict(X_test_proc)
    test_acc.append(accuracy_score(y_test, preds))
    test_f1.append(f1_score(y_test, preds, zero_division=0))

best_k   = k_values[np.argmax(test_f1)]
best_f1v = max(test_f1)
print(f'k tối ưu (F1): k={best_k}  →  F1={best_f1v:.4f}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_values, train_acc, 'o-', color='#3498db', label='Train Accuracy')
axes[0].plot(k_values, test_acc,  's-', color='#e74c3c', label='Test Accuracy')
axes[0].axvline(x=best_k, color='gray', linestyle='--', alpha=0.7)
axes[0].set_xlabel('k (số láng giềng)')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('kNN: Train vs Test Accuracy theo k', fontweight='bold')
axes[0].legend()

axes[1].plot(k_values, test_f1, 'D-', color='#2ecc71', label='Test F1-Score')
axes[1].axvline(x=best_k, color='gray', linestyle='--', alpha=0.7, label=f'k={best_k}')
axes[1].scatter([best_k], [best_f1v], color='#e74c3c', zorder=5, s=80)
axes[1].set_xlabel('k (số láng giềng)')
axes[1].set_ylabel('F1-Score')
axes[1].set_title('kNN: Test F1-Score theo k', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()


---
## Bước 11: Tổng kết


In [ ]:
# ---- Bảng tổng kết cuối cùng ----
print('='*65)
print('  TỔNG KẾT KẾT QUẢ — HOMEWORK 1: Data Preprocessing & kNN')
print('='*65)
print()
print('📦 Dataset: Telco Customer Churn (Synthetic + UCI-based)')
print(f'   Mẫu: {len(df):,}  |  Features: {X_raw.shape[1]}  |  Nhãn: Churn (Yes/No)')
print()
print('🔧 Các bước Data Cleaning đã thực hiện:')
print('   ✔ Xóa 30 dòng duplicate')
print('   ✔ Chuẩn hoá casing inconsistent (gender, ...)')
print('   ✔ Xử lý missing values (Median/Mode imputation)')
print('   ✔ Xử lý invalid values (tenure<0, MonthlyCharges>500)')
print()
print('⚙️ Feature Engineering:')
print('   ✔ Drop customerID (text, không có giá trị dự đoán)')
print('   ✔ ContractRank (ordinal encoding: Month-to-month=0, ...)')
print('   ✔ ChargesPerMonth = TotalCharges / tenure')
print('   ✔ NumServices (đếm dịch vụ thêm)')
print('   ✔ TenureGroup (bin tenure thành 4 nhóm)')
print()
print('🔄 Preprocessing Pipeline:')
print('   ✔ Numerical  → Median Impute → StandardScaler')
print('   ✔ Binary     → Mode Impute')
print('   ✔ Categorical → Mode Impute → One-Hot Encoding')
print()
print('📊 Kết quả các mô hình (Test Set):')
print(f'  {"Mô hình":<22} {"Accuracy":>9} {"Precision":>10} {"Recall":>8} {"F1":>8}')
print('  ' + '-'*60)
for name, v in results.items():
    print(f'  {name:<22} {v["Accuracy"]:>9.4f} {v["Precision"]:>10.4f} {v["Recall"]:>8.4f} {v["F1"]:>8.4f}')
print()

best_model = max(results.items(), key=lambda x: x[1]['F1'])
print(f'🏆 Mô hình tốt nhất (F1): {best_model[0]}')
print(f'   Accuracy = {best_model[1]["Accuracy"]:.4f}')
print(f'   F1-Score = {best_model[1]["F1"]:.4f}')
print(f'\n✅ kNN tối ưu: k={best_k}  →  F1={best_f1v:.4f}')
